In [2]:
import pandas as pd
from hypernetx import Hypergraph

def build_hypergraph(edge_to_vertices):
    """
    PAPER: Build X=(V,𝓔) from explicit hyperedge family 𝓔(X),
    edge_to_vertices = { e_id : set_of_vertices }.
    """
    rows = []
    for e_id, verts in edge_to_vertices.items():
        for v in verts:
            rows.append({"edges": e_id, "nodes": v, "weight": 1, "misc_properties": None})
    return Hypergraph(pd.DataFrame(rows))

def edge_sets(H):
    """Return 𝓔(H) as dict: e -> set of vertices in e."""
    return {e: set(verts) for e, verts in H.incidence_dict.items()}

def print_hyperedges(H, title="Hyperedges"):
    print(title)
    for e, verts in edge_sets(H).items():
        print(f"  {e}: {sorted(verts)}")

def minimize_dual(H_star):
    """
    PAPER: M(H*) is obtained by deleting any hyperedge contained in another
    (keep inclusion-maximal hyperedges only).
    """
    return H_star.restrict_to_edges(H_star.toplexes())

def remove_size1_edges(H):
    """
    PAPER: Enforce |e| >= 2 for all hyperedges e ∈ 𝓔(H),
    since singleton hyperedges are not useful for our leaf logic.
    """
    E = edge_sets(H)
    keep = [e for e, verts in E.items() if len(verts) >= 2]
    return H.restrict_to_edges(keep)

In [3]:
# Toy H: vertices stand in for maximal cliques of G (K1,K2,...)
# Hyperedges stand in for original vertices of G (each lists cliques containing it).
E_H = {
    "v1": {"K1", "K2", "K3"},
    "v2": {"K2", "K3"},
    "v3": {"K3", "K4"},
    "v4": {"K1", "K3"},
    "v5": {"K4"},  # singleton (to be removed later in M(H*))
}

H = build_hypergraph(E_H)
H_star = H.dual()

H_star_hat = minimize_dual(H_star)
H_star_hat = remove_size1_edges(H_star_hat)

print_hyperedges(H_star_hat, "M(H*) (minimized dual, with |e|>=2):")

M(H*) (minimized dual, with |e|>=2):
  K3: ['v1', 'v2', 'v3', 'v4']
  K4: ['v3', 'v5']


Lets calculate degree

In [4]:
def vertex_incidence_degree(H):
    """
    PAPER: For a vertex x ∈ V(H), define deg_H(x) = |{ e ∈ 𝓔(H) : x ∈ e }|.
    """
    deg = {}
    for e, verts in H.incidence_dict.items():
        for x in verts:
            deg[x] = deg.get(x, 0) + 1
    return deg

deg = vertex_incidence_degree(H_star_hat)
deg

{'v1': 1, 'v2': 1, 'v3': 2, 'v4': 1, 'v5': 1}

Lets classify alpha leaves and non-alpha leaves or parents

In [6]:
def classify_alpha_leaves_and_parents(H_star_hat):
    """
    PAPER DEFINITIONS (your phrasing):
      - parent vertex: incident to >= 2 hyperedges in Ĥ*
      - α-leaf vertex: incident to exactly 1 hyperedge in Ĥ*
    """
    deg = vertex_incidence_degree(H_star_hat)

    # create sorted lists of α-leaves and parents
    alpha_leaves = sorted([x for x, d in deg.items() if d == 1], key=str) # sort by string for consistent order
    parents      = sorted([x for x, d in deg.items() if d >= 2], key=str) 

    return alpha_leaves, parents, deg

alpha_leaves, parents, deg = classify_alpha_leaves_and_parents(H_star_hat)

print("α-leaves (deg=1):", alpha_leaves)
print("parents  (deg>=2):", parents)
print("degree map:", dict(sorted(deg.items(), key=lambda kv: str(kv[0])))) # sort degree map by vertex name for consistent order

α-leaves (deg=1): ['v1', 'v2', 'v4', 'v5']
parents  (deg>=2): ['v3']
degree map: {'v1': 1, 'v2': 1, 'v3': 2, 'v4': 1, 'v5': 1}


In [7]:
def incident_edges_of_vertex(H, x):
    """
    PAPER: return { e ∈ 𝓔(H) : x ∈ e }.
    """
    inc = []
    for e, verts in H.incidence_dict.items():
        if x in verts:
            inc.append(e)
    return inc

for x in alpha_leaves:
    inc_edges = incident_edges_of_vertex(H_star_hat, x)
    print(f"α-leaf {x} lies in hyperedge(s): {inc_edges}")

α-leaf v1 lies in hyperedge(s): ['K3']
α-leaf v2 lies in hyperedge(s): ['K3']
α-leaf v4 lies in hyperedge(s): ['K3']
α-leaf v5 lies in hyperedge(s): ['K4']
